# 🏸 Sports Video Player Action Recognition & Court Zone Tracking
**CNN (ResNet-18, per frame) + LSTM (temporal) on a UCF101 sports subset — 100% free on Google Colab.**

Pipeline:
1. Download UCF101 from Kaggle and cache 10 sports classes (→ 4 actions: *serving, running, smashing, volleying*).
2. Train the CNN-LSTM with leakage-free, group-based splits.
3. Evaluate accuracy (confusion matrix, per-class F1, TTA) and **frame rate**.
4. Run the full video pipeline: action recognition + player detection/tracking + court-zone analytics.

> **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision; print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| CUDA', torch.cuda.is_available())

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/LuShadowX/sports-action-recognition.git"
import os
if not os.path.exists("sports-action-recognition"):
    !git clone -q $REPO_URL sports-action-recognition
%cd sports-action-recognition
!pip install -q -r requirements.txt

## 2. Download UCF101 (Kaggle) and build the frame cache
`kagglehub` downloads public datasets without setup. If Kaggle asks for credentials, add `KAGGLE_USERNAME` / `KAGGLE_KEY`
in Colab **Secrets** (🔑 icon). The full dataset is ~6.5 GB; only 10 sports classes (~1,400 clips) are cached.

In [ ]:
import os, kagglehub
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError:
    pass

DATA_ROOT = None
for slug in ["matthewjansen/ucf101-action-recognition", "pevogam/ucf101"]:
    try:
        DATA_ROOT = kagglehub.dataset_download(slug); print("✅", slug, "->", DATA_ROOT); break
    except Exception as e:
        print("⚠️", slug, "failed:", str(e)[:200])
print("DATA_ROOT =", DATA_ROOT)

*Fallback (only if both Kaggle mirrors fail): official UCF101 archive.*

In [ ]:
# Uncomment to use the official source instead of Kaggle
# !wget -q --no-check-certificate https://www.crcv.ucf.edu/data/UCF101/UCF101.rar -O /content/UCF101.rar
# !apt-get -qq install -y unrar && unrar x -inul /content/UCF101.rar /content/
# DATA_ROOT = "/content/UCF-101"

In [ ]:
!python prepare_data.py --data-root "$DATA_ROOT" --workers 4

## 3. Train the CNN-LSTM
Defaults (`configs/default.yaml`): ResNet-18 (ImageNet, last stage fine-tuned) → 2-layer BiLSTM(256) → temporal attention → classifier;
16 frames @160px, AdamW, warmup+cosine, label smoothing, AMP, early stopping. ~1 min/epoch on a T4.

In [ ]:
!python train.py --set train.epochs=25

In [ ]:
from IPython.display import Image, display
import json
display(Image("runs/cnn_lstm/training_curves.png", width=850))
display(Image("runs/cnn_lstm/confusion_matrix.png", width=520))
m = json.load(open("runs/cnn_lstm/test_metrics.json"))
print(f"Test accuracy {m['accuracy']*100:.2f}% | macro-F1 {m['macro_f1']*100:.2f}% | {m['model_frames_per_s']:.0f} frames/s")

## 4. Evaluation with test-time augmentation (4 views)

In [ ]:
!python evaluate.py --checkpoint runs/cnn_lstm/best.pt --tta
display(Image("runs/cnn_lstm/confusion_matrix_test_tta.png", width=520))

## 5. Frame-rate benchmark (model-only + real-video streaming)

In [ ]:
import json, glob
man = json.load(open("data/cache/manifest.json"))
test_clips = [r for r in man["records"] if r["split"] == "test"]
DEMO = next(r["video_path"] for r in test_clips if r["fine_class"] == "TennisSwing")
print("Demo clip:", DEMO)
!python benchmark_fps.py --checkpoint runs/cnn_lstm/best.pt --video "$DEMO"

## 6. Full pipeline: action + player tracking + court zones
Without calibration the zones are a 3×3 grid on the image. For real match footage, read the 4 court corners from the
first frame (far-left, far-right, near-right, near-left) and pass `--court tennis --court-corners "x,y;x,y;x,y;x,y" --court-only`.

In [ ]:
!python infer_video.py --checkpoint runs/cnn_lstm/best.pt --video "$DEMO" --out outputs/demo.mp4
!ffmpeg -y -loglevel error -i outputs/demo.mp4 -vcodec libx264 -pix_fmt yuv420p outputs/demo_h264.mp4
from IPython.display import HTML; from base64 import b64encode
data = b64encode(open("outputs/demo_h264.mp4", "rb").read()).decode()
HTML(f'<video width=640 controls autoplay loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

In [ ]:
s = json.load(open("outputs/demo.summary.json"))
print(json.dumps({k: s[k] for k in ["clip_prediction", "action_timeline", "players", "speed"]}, indent=2)[:3000])
display(Image("outputs/demo.heatmap.png", width=320))

### Use your own match video
Upload a clip, show its first frame with a pixel grid, read the four court corners, then run the pipeline.

In [ ]:
# from google.colab import files; up = files.upload(); MY_VIDEO = next(iter(up))
# import cv2, matplotlib.pyplot as plt
# ok, f = cv2.VideoCapture(MY_VIDEO).read(); plt.figure(figsize=(14, 8)); plt.imshow(f[..., ::-1]); plt.grid(alpha=.4); plt.show()
# !python infer_video.py --checkpoint runs/cnn_lstm/best.pt --video "$MY_VIDEO" --court tennis \
#     --court-corners "410,180;870,180;1150,650;130,650" --court-only --max-players 4

## 7. (Optional) Experiments for the report
* **Fine labels** (10 UCF101 classes) vs coarse (4 actions)
* **Backbone speed/accuracy trade-off**: MobileNetV3 vs ResNet-18
* **Frozen CNN** vs fine-tuned last stage

In [ ]:
# !python train.py --set data.label_mode=fine --set train.output_dir=runs/fine
# !python train.py --set model.backbone=mobilenet_v3_large --set train.output_dir=runs/mnv3
# !python train.py --set model.trainable_backbone_stages=0 --set train.output_dir=runs/frozen

In [ ]:
# Save results to Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/sports_ar && cp -r runs outputs /content/drive/MyDrive/sports_ar/